In [ ]:
project_path = "/home/jupyter"
import os
import sys

sys.path.append(project_path)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from google.cloud import bigquery

from fintrans_toolbox.src import bq_utils as bq
from fintrans_toolbox.src import table_utils as t


client = bigquery.Client()

In [ ]:
UK_spending_by_country = '''SELECT time_period_value, destination_country, SUM(spend) AS total FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` where time_period = 'Quarter' and merchant_channel = 'Online' and cardholder_origin_country = 'All' and cardholder_origin = 'UNITED KINGDOM' and destination_country != 'UNITED KINGDOM' GROUP BY destination_country, 
time_period_value ORDER BY time_period_value, total DESC'''
df_by_country = bq.read_bq_table_sql(client, UK_spending_by_country)
df_by_country

In [ ]:
UK_spending_by_mccgoods = '''SELECT 
    mcc, 
    SUM(spend) AS total_spend, 
    (SUM(spend) * 100.0 / (SELECT SUM(spend)  
                                          FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` 
                                          WHERE mcc IN ( 
        'ANTIQUE SHOPS', 
        'ARTIST/CRAFT SHOPS',
        'ART DEALERS & GALLERIES',
        'AUTOMOTIVE PARTS STORES', 
        'AUTOMOTIVE TIRE STORES', 
        'BAKERIES', 
        'BOOKS/PERIODICALS/NEWSPAPERS',
        'BOOK STORES', 
        'CANDY/NUT/CONFECTION STORES',
        'CAR & TRUCK DEALERS/NEW/USED', 
        'CAR & TRUCK DEALERS/USED ONLY', 
        'CAMERA & PHOTO SUPPLY STORES', 
        'COSMETIC STORES', 
        'DEPARTMENT STORES', 
        'ELECTRONICS STORES', 
        'FABRIC STORES', 
        'FLOOR COVERING STORES', 
        'FLORISTS', 
        'GLASS/PAINT/WALLPAPER STORES', 
        'GIFT, CARD, NOVELTY STORES', 
        'HOUSEHOLD APPLIANCE STORES', 
        'JEWELRY STORES', 
        'LUGGAGE/LEATHER STORES'
        'LUMBER/BUILD. SUPPLY STORES', 
        'MISC FOOD STORES - DEFAULT',
        'MISC GENERAL MERCHANDISE', 
        'MISC HOME FURNISHING SPECIALTY', 
        'MISC SPECIALTY RETAIL', 
        'MOBILE HOME DEALERS', 
        'MOTOR VEHICLE SUPPLY/NEW PARTS', 
        'MOTORCYCLE DEALERS', 
        'MUSIC STORES/PIANOS', 
        'PET STORES/FOOD & SUPPLY', 
        'PHOTO STUDIOS', 
        'PLUMBING/HEATING EQUIPMENT',
        'PRECIOUS STONES/METALS/JEWELRY',
        'RECORD STORES',
        'RELIGIOUS GOODS STORES',
        'ROOFING/SIDING/SHEET METAL'
        'SHOE STORES',
        'SPORTING GOODS STORES',
        'SPORTS/RIDING APPAREL STORES',
        'STAMP & COIN STORES',
        'STATIONERY STORES',
        'STATIONERY/OFFICE SUPPLIES',
        'TENT AND AWNING SHOPS',
        'TELECOMMUNICATION EQUIPMENT',
        'UNIFORMS & COMMERCIAL CLOTHING',
        'VEHICLE RENTAL',
        'WIG AND TOUPEE STORES',
        'WOMENS READY TO WEAR STORES',
        'WRECKING SALVAGE YARDS',
        'HOME SUPPLY WAREHOUSE STORES',
'INDUSTRIAL SUPPLIES - DEF',
'MISC AUTO DEALERS - DEFAULT',
'MOTOR HOME DEALERS',
'PLUMBING/HEATING EQUIPMENT',
'SERVICE STATIONS',
'VIDEO AMUSEMENT GAME SUPPLY',
'BICYCLE SHOPS/SALES/SERVICE',
'SWIMMING POOLS/SALES/SERV'
) ) ) AS b2b_goods_percentage 
FROM  
    `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` 
WHERE  
    mcc IN ( 
'ANTIQUE SHOPS', 
        'ARTIST/CRAFT SHOPS',
        'ART DEALERS & GALLERIES',
        'AUTOMOTIVE PARTS STORES', 
        'AUTOMOTIVE TIRE STORES', 
        'BAKERIES', 
        'BOOKS/PERIODICALS/NEWSPAPERS',
        'BOOK STORES', 
        'CANDY/NUT/CONFECTION STORES',
        'CAR & TRUCK DEALERS/NEW/USED', 
        'CAR & TRUCK DEALERS/USED ONLY', 
        'CAMERA & PHOTO SUPPLY STORES', 
        'COSMETIC STORES', 
        'DEPARTMENT STORES', 
        'ELECTRONICS STORES', 
        'FABRIC STORES', 
        'FLOOR COVERING STORES', 
        'FLORISTS', 
        'GLASS/PAINT/WALLPAPER STORES', 
        'GIFT, CARD, NOVELTY STORES', 
        'HOUSEHOLD APPLIANCE STORES', 
        'JEWELRY STORES', 
        'LUGGAGE/LEATHER STORES'
        'LUMBER/BUILD. SUPPLY STORES', 
        'MISC FOOD STORES - DEFAULT',
        'MISC GENERAL MERCHANDISE', 
        'MISC HOME FURNISHING SPECIALTY', 
        'MISC SPECIALTY RETAIL', 
        'MOBILE HOME DEALERS', 
        'MOTOR VEHICLE SUPPLY/NEW PARTS', 
        'MOTORCYCLE DEALERS', 
        'MUSIC STORES/PIANOS', 
        'PET STORES/FOOD & SUPPLY', 
        'PHOTO STUDIOS', 
        'PLUMBING/HEATING EQUIPMENT',
        'PRECIOUS STONES/METALS/JEWELRY',
        'RECORD STORES',
        'RELIGIOUS GOODS STORES',
        'ROOFING/SIDING/SHEET METAL'
        'SHOE STORES',
        'SPORTING GOODS STORES',
        'SPORTS/RIDING APPAREL STORES',
        'STAMP & COIN STORES',
        'STATIONERY STORES',
        'STATIONERY/OFFICE SUPPLIES',
        'TENT AND AWNING SHOPS',
        'TELECOMMUNICATION EQUIPMENT',
        'UNIFORMS & COMMERCIAL CLOTHING',
        'VEHICLE RENTAL',
        'WIG AND TOUPEE STORES',
        'WOMENS READY TO WEAR STORES',
        'WRECKING SALVAGE YARDS',
        'HOME SUPPLY WAREHOUSE STORES',
'INDUSTRIAL SUPPLIES - DEF',
'MISC AUTO DEALERS - DEFAULT',
'MOTOR HOME DEALERS',
'PLUMBING/HEATING EQUIPMENT',
'SERVICE STATIONS',
'VIDEO AMUSEMENT GAME SUPPLY',
'BICYCLE SHOPS/SALES/SERVICE',
'SWIMMING POOLS/SALES/SERV') 
and time_period = 'Quarter' 
and merchant_channel = 'Online' 
and cardholder_origin_country = 'All' 
and cardholder_origin = 'UNITED KINGDOM' 
and destination_country != 'UNITED KINGDOM'
GROUP BY  
    mcc
ORDER BY  
    total_spend DESC'''
df_by_mccgoods = bq.read_bq_table_sql(client, UK_spending_by_mccgoods)     
df_by_mccgoods

In [ ]:
df_by_mccgoods.to_csv('UK_spending_by_mccgoods.csv')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure the data is in the correct format
# Assuming df_by_mccgoods is your DataFrame from the SQL query result

# Sort data by 'total_spend' for better visualization
df_by_mccgoods = df_by_mccgoods.sort_values(by='total_spend', ascending=False)

# Set the plot size for readability
plt.figure(figsize=(12, 8))

# Create a bar chart with total spend
sns.barplot(x='total_spend', y='mcc', data=df_by_mccgoods, palette='viridis')

# Add labels and title
plt.xlabel('Total Spend (£)', fontsize=14)
plt.ylabel('Merchant Category Code (MCC)', fontsize=14)
plt.title('Total Spend by MCC (Online)', fontsize=16)

# Add a second axis to show B2B percentage
# If you want to visualize both total spend and B2B percentage in a dual-axis chart, you can do something like this:

fig, ax1 = plt.subplots(figsize=(12, 8))

# Plot total spend
sns.barplot(x='total_spend', y='mcc', data=df_by_mccgoods, palette='Blues', ax=ax1)
ax1.set_xlabel('Total Spend (£)', fontsize=14)
ax1.set_ylabel('Merchant Category Code (MCC)', fontsize=14)
ax1.set_title('Total Spend and B2B Goods Percentage by MCC (Online Abroad)', fontsize=16)

# Create a second axis for B2B percentage
ax2 = ax1.twiny()
sns.lineplot(x='b2b_goods_percentage', y='mcc', data=df_by_mccgoods, color='red', marker='o', ax=ax2)
ax2.set_xlabel('B2B Goods Percentage (%)', fontsize=14, color='red')
ax2.tick_params(axis='x', labelcolor='red')

# Display the plot
plt.show()

In [ ]:
UK_spending_by_mccgoods_yearly = '''
SELECT 
    CAST(SUBSTR(time_period_value, 1, 4) AS INT64) AS year,  -- Extract the first 4 characters (year) from time_period_value
    SUM(spend) AS mcc_goods_total  -- Sum the total spend for each year
FROM  
    `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` 
WHERE  
    mcc IN ( 
        'ANTIQUE SHOPS', 
        'ART DEALERS & GALLERIES',
        'ARTIST/CRAFT SHOPS',
        'AUTOMOTIVE PARTS STORES', 
        'AUTOMOTIVE TIRE STORES', 
        'BAKERIES', 
        'BOOKS/PERIODICALS/NEWSPAPERS',
        'BOOK STORES', 
        'CANDY/NUT/CONFECTION STORES',
        'CAR & TRUCK DEALERS/NEW/USED', 
        'CAR & TRUCK DEALERS/USED ONLY', 
        'CAMERA & PHOTO SUPPLY STORES', 
        'COSMETIC STORES', 
        'DEPARTMENT STORES', 
        'ELECTRONICS STORES', 
        'FABRIC STORES', 
        'FLOOR COVERING STORES', 
        'FLORISTS', 
        'GLASS/PAINT/WALLPAPER STORES', 
        'GIFT, CARD, NOVELTY STORES', 
        'HOUSEHOLD APPLIANCE STORES', 
        'JEWELRY STORES', 
        'LUGGAGE/LEATHER STORES'
        'LUMBER/BUILD. SUPPLY STORES', 
        'MISC FOOD STORES - DEFAULT',
        'MISC GENERAL MERCHANDISE', 
        'MISC HOME FURNISHING SPECIALTY', 
        'MISC SPECIALTY RETAIL', 
        'MOBILE HOME DEALERS', 
        'MOTOR VEHICLE SUPPLY/NEW PARTS', 
        'MOTORCYCLE DEALERS', 
        'MUSIC STORES/PIANOS', 
        'PET STORES/FOOD & SUPPLY', 
        'PHOTO STUDIOS', 
        'PLUMBING/HEATING EQUIPMENT',
        'PRECIOUS STONES/METALS/JEWELRY',
        'RECORD STORES',
        'RELIGIOUS GOODS STORES',
        'ROOFING/SIDING/SHEET METAL'
        'SHOE STORES',
        'SPORTING GOODS STORES',
        'SPORTS/RIDING APPAREL STORES',
        'STAMP & COIN STORES',
        'STATIONERY STORES',
        'STATIONERY/OFFICE SUPPLIES',
        'TENT AND AWNING SHOPS',
        'TELECOMMUNICATION EQUIPMENT',
        'UNIFORMS & COMMERCIAL CLOTHING',
        'VEHICLE RENTAL',
        'WIG AND TOUPEE STORES',
        'WOMENS READY TO WEAR STORES',
        'WRECKING SALVAGE YARDS',
        'HOME SUPPLY WAREHOUSE STORES',
'INDUSTRIAL SUPPLIES - DEF',
'MISC AUTO DEALERS - DEFAULT',
'MOTOR HOME DEALERS',
'PLUMBING/HEATING EQUIPMENT',
'SERVICE STATIONS',
'VIDEO AMUSEMENT GAME SUPPLY',
'BICYCLE SHOPS/SALES/SERVICE',
'SWIMMING POOLS/SALES/SERV'
    ) 
    AND time_period = 'Quarter' 
    AND merchant_channel = 'Online' 
    and cardholder_origin_country = 'All' 
and cardholder_origin = 'UNITED KINGDOM' 
and destination_country != 'UNITED KINGDOM'
    AND CAST(SUBSTR(time_period_value, 1, 4) AS INT64) BETWEEN 2019 AND 2024  -- Filter by year range
GROUP BY  
    year
ORDER BY  
    year'''

df_by_mccgoods_yearly = bq.read_bq_table_sql(client, UK_spending_by_mccgoods_yearly)
df_by_mccgoods_yearly


In [ ]:
#rgba (which stands for red, green, blue, and alpha transparency), here are a few examples:

#Light Green:
rgba(144, 238, 144, 0.6)
This represents a light green color with 60% opacity.

#Dark Green:
rgba(0, 128, 0, 0.6)
This is a darker green with 60% opacity.

#Forest Green:
rgba(34, 139, 34, 0.6)
This is a deeper forest green with 60% opacity.

#Lime Green:
rgba(50, 205, 50, 0.6)
This is a brighter lime green with 60% opacity.

#Olive Green:
rgba(128, 128, 0, 0.6)
This is an olive green with 60% opacity.


In [ ]:
import plotly.graph_objects as go
import pandas as pd

# Define the data
data = {
    'year': [2019, 2020, 2021, 2022, 2023, 2024],
    'mcc_goods_total': [7.423720e+09, 8.300689e+09, 9.123387e+09, 3.315036e+09, 3.705416e+09, 2.101098e+09
]
}

# Create the DataFrame
df = pd.DataFrame(data)

# Create the bar trace for 'mcc_goods_total'
bar_trace = go.Bar(
    x=df['year'],
    y=df['mcc_goods_total'],
    name='MCC Goods Total',
    text=df['mcc_goods_total'],  # Display value on top of bars
    hoverinfo='text+x+y',  # Display info on hover
    marker=dict(color='rgba(50, 205, 50, 0.6)')  # Bar color - Green
)

# Create the line trace for 'mcc_goods_total'
line_trace = go.Scatter(
    x=df['year'],
    y=df['mcc_goods_total'],
    mode='lines+markers',  # Line with markers
    name='MCC Goods Total Line',
    line=dict(color='rgb(0, 123, 255)', width=2),  # Line color and width
    marker=dict(color='rgb(0, 123, 255)', size=6)  # Marker color and size
)

# Combine the bar and line traces
fig = go.Figure(data=[bar_trace, line_trace])

# Update the layout of the chart
fig.update_layout(
    title="MCC Goods Online Abroad Total by Year",
    xaxis_title="Year",
    yaxis_title="MCC Goods Total (in GBP)",
    template="plotly_dark",
    xaxis=dict(tickmode='array', tickvals=df['year']),  # Ensures that x-axis is properly labeled
    yaxis=dict(title="Amount in GBP", showgrid=True, zeroline=False),
    showlegend=True
)

# Display the chart
fig.show()


In [ ]:
import plotly.graph_objects as go

# Data for 'mcc_goods_total'
df_mcc = pd.DataFrame({
    'year': [2019, 2020, 2021, 2022, 2023, 2024],
    'mcc_goods_total': [7.423720e+09, 8.300689e+09, 9.123387e+09, 3.315036e+09, 3.705416e+09, 2.101098e+09]
})

# Create the bar trace
bar_trace = go.Bar(
    x=df_mcc['year'],
    y=df_mcc['mcc_goods_total'] / 1e9,  # Convert values to billions
    name='MCC Goods Abroad Online Total',
    text=(df_mcc['mcc_goods_total'] / 1e9).round(2),  # Show values in billions with 2 decimal places
    hoverinfo='text+x+y',  # Display info on hover
    marker=dict(color='green'),  # Set bar color to green
    orientation='v'
)

# Create the line trace for the same data
line_trace = go.Scatter(
    x=df_mcc['year'],
    y=df_mcc['mcc_goods_total'] / 1e9,  # Convert values to billions
    mode='lines+markers',
    name='MCC Goods Abroad Online Total (Line)',
    line=dict(color='red', width=2),  # Line color and thickness
    marker=dict(color='red')  # Marker color
)

# Plotting the Bar and Line Chart
fig = go.Figure(data=[bar_trace, line_trace])

fig.update_layout(
    title="MCC Goods Abroad Online Total (Billions GBP)",
    xaxis_title="Year",
    yaxis_title="Amount in Billions GBP",  # Updated y-axis title
    template="plotly_dark",
    xaxis=dict(tickmode='array', tickvals=df_mcc['year']),
    yaxis=dict(
        title="Amount in Billions GBP",
        showgrid=True,
        zeroline=False,
        tickformat='.0f'  # Format y-axis to show 2 decimal places
    ),
    showlegend=True
)

# Display the combined bar and line chart
fig.show()


In [ ]:
# UK Spend Abroad Total (Visa)
   year  abroad_spend_all
0  2019      4.162683e+10
1  2020      3.659158e+10
2  2021      3.586756e+10
3  2022      3.485071e+10
4  2023      3.577281e+10
5  2024      2.890089e+10

#UK Spend Domestic Total (Visa)
   year         spend
0  2019  5.155348e+11
1  2020  5.218997e+11
2  2021  5.509458e+11
3  2022  5.316535e+11
4  2023  5.108432e+11
5  2024  3.786755e+11

In [ ]:
import pandas as pd

# Data for UK Spend Abroad Total (Visa)
df_abroad_spend = pd.DataFrame({
    'year': [2019, 2020, 2021, 2022, 2023, 2024],
    'abroad_spend_all': [4.162683e+10, 3.659158e+10, 3.586756e+10, 3.485071e+10, 3.577281e+10, 2.890089e+10]
})

# Data for UK Spend Domestic Total (Visa)
df_domestic_spend = pd.DataFrame({
    'year': [2019, 2020, 2021, 2022, 2023, 2024],
    'spend': [5.155348e+11, 5.218997e+11, 5.509458e+11, 5.316535e+11, 5.108432e+11, 3.786755e+11]
})

# Merging the two dataframes on 'year' column
df_combined = pd.merge(df_abroad_spend, df_domestic_spend, on='year')

# Creating a new column for the total spending
df_combined['total_spending_visa'] = df_combined['abroad_spend_all'] + df_combined['spend']

# Display the combined dataframe with total spending
print(df_combined[['year', 'total_spending_visa']])


In [ ]:
import pandas as pd

# Data for mcc_goods_total
df_mcc_goods = pd.DataFrame({
    'year': [2019, 2020, 2021, 2022, 2023, 2024],
    'mcc_goods_total': [7.423720e+09, 8.300689e+09, 9.123387e+09, 3.315036e+09, 3.705416e+09, 2.101098e+09]
})

# Data for total_spending_visa
df_total_spending_visa = pd.DataFrame({
    'year': [2019, 2020, 2021, 2022, 2023, 2024],
    'total_spending_visa': [5.571616e+11, 5.584913e+11, 5.868134e+11, 5.665042e+11, 5.466160e+11, 4.075764e+11]
})

# Merging the two dataframes on the 'year' column
df_combined = pd.merge(df_mcc_goods, df_total_spending_visa, on='year')

# Calculating the Goods Online Abroad Ratio
df_combined['Goods_Online_Abroad_Ratio'] = (df_combined['mcc_goods_total'] / df_combined['total_spending_visa']) * 100

# Display the result
print(df_combined[['year', 'Goods_Online_Abroad_Ratio']])


In [ ]:
#Total Value of Purchases UK Finance
year     total spend
0  2019  8.03E+11
1  2020  8.02E+11
2  2021  8.74E+11
3  2022  9.65E+11
4  2023  1.02E+12

In [ ]:
#Calculate UK Finance Total Goods Abroad Online Spending

import pandas as pd

# Data for Total UK Finance Spending
df_total_uk_finance = pd.DataFrame({
    'year': [2019, 2020, 2021, 2022, 2023],
    'total_spend': [8.03e+11, 8.02e+11, 8.74e+11, 9.65e+11, 1.02e+12]
})

# Data for Total Visa Spending
df_total_spending_visa = pd.DataFrame({
    'year': [2019, 2020, 2021, 2022, 2023, 2024],
    'total_spending_visa': [5.571616e+11, 5.584913e+11, 5.868134e+11, 5.665042e+11, 5.466160e+11, 4.075764e+11]
})

# Data for Goods Online Abroad Ratio (already calculated earlier)
df_goods_online_abroad_ratio = pd.DataFrame({
    'year': [2019, 2020, 2021, 2022, 2023, 2024],
    'Goods_Online_Abroad_Ratio': [1.33, 1.49, 1.56, 0.58, 0.68, 0.51]
})

# Merging the dataframes on the 'year' column
df_combined = pd.merge(df_total_uk_finance, df_total_spending_visa, on='year', how='inner')
df_combined = pd.merge(df_combined, df_goods_online_abroad_ratio, on='year', how='inner')

# Calculating the UK Finance Online Abroad Total Spending
df_combined['UK_Finance_Goods_Online_Abroad_Total_Spending'] = (
    (df_combined['total_spend'] * df_combined['Goods_Online_Abroad_Ratio']
)/100)

# Display the result
print(df_combined[['year', 'UK_Finance_Goods_Online_Abroad_Total_Spending']])


In [ ]:
import plotly.graph_objects as go

# Data for 'mcc_goods_total'
df_mcc = pd.DataFrame({
    'year': [2019, 2020, 2021, 2022, 2023],
    'mcc_goods_total': [1.067990e+10, 1.194980e+10, 1.363440e+10, 5.597000e+09, 6.936000e+09]
})

# Create the bar trace
bar_trace = go.Bar(
    x=df_mcc['year'],
    y=df_mcc['mcc_goods_total'] / 1e9,  # Convert values to billions
    name='Goods Abroad Online Total',
    text=(df_mcc['mcc_goods_total'] / 1e9).round(2),  # Show values in billions with 2 decimal places
    hoverinfo='text+x+y',  # Display info on hover
    marker=dict(color='green'),  # Set bar color to green
    orientation='v'
)

# Create the line trace for the same data
line_trace = go.Scatter(
    x=df_mcc['year'],
    y=df_mcc['mcc_goods_total'] / 1e9,  # Convert values to billions
    mode='lines+markers',
    name='Goods Abroad Online Total (Line)',
    line=dict(color='red', width=2),  # Line color and thickness
    marker=dict(color='red')  # Marker color
)

# Plotting the Bar and Line Chart
fig = go.Figure(data=[bar_trace, line_trace])

fig.update_layout(
    title="UK Finance Goods Abroad Online Total (Billions GBP)",
    xaxis_title="Year",
    yaxis_title="Amount in Billions GBP",  # Updated y-axis title
    template="plotly_dark",
    xaxis=dict(tickmode='array', tickvals=df_mcc['year']),
    yaxis=dict(
        title="Amount in Billions GBP",
        showgrid=True,
        zeroline=False,
        tickformat='.0f'  # Format y-axis to show 0 decimal places
    ),
    showlegend=True
)

# Display the combined bar and line chart
fig.show()

In [ ]:
direct_marketing = '''SELECT time_period_value, destination_country, SUM(spend) AS total FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` where time_period = 'Quarter' and merchant_channel = 'Online' and cardholder_origin_country = 'All' and cardholder_origin = 'UNITED KINGDOM' and destination_country != 'UNITED KINGDOM' GROUP BY destination_country, 
time_period_value ORDER BY time_period_value, total DESC'''